# Final real-MNIST digit `1` geometry feasibility experiment

**Research question:** Can our *frozen projector-aware noise-scale selector* produce a numerically credible pixel-space local dimension and tangent geometry for **one clean, held-out real MNIST digit `1`**, using a **pretrained class-conditional diffusion score model**?

**Checkpoint:** [`harveymannering/mnist-ddpm`](https://huggingface.co/harveymannering/mnist-ddpm) (28×28 MNIST, 11 class embeddings; class `1`). **No model training.** This is **not** a run of the *official* LHSD baseline; the comparator here is the earlier frozen **LHSD-inspired transition-mass-only selector**.

### What constitutes evidence?

1. Consistent score-Jacobian numerics (central finite difference vs first-order adjoint, with no unsupported double backward).
2. A contiguous set of noise scales passing **pre-frozen**, label-independent criteria. Either selector may **ABSTAIN**.
3. A dimension estimate with probe uncertainty (not a true-dimension error—**MNIST ground truth is unknown**).
4. **Independent post-selection** tangent-filter checks using translations, local clean digit-`1` variations, and random controls. These are heuristic proxies, not known true MNIST tangents.

The notebook has a **low-compute pilot**, then an optional expanded run. Choose **Runtime → Change runtime type → T4 GPU** if available (CPU may be much slower). Run cells top to bottom. The last cell can package your results.

**Important:** All algorithm thresholds are fixed from the earlier synthetic experiment. We will not change them to force a favorable MNIST result. A rejected estimate is an informative outcome, not a reason to fabricate a dimension.

## 1 · Install only the required packages
This notebook is self-contained and writes all its source code below. It does not depend on any earlier Colab.

In [ ]:
!pip -q install "diffusers==0.38.0" "huggingface_hub>=0.28" "safetensors>=0.4" "accelerate>=1.0"

## 2 · Write the complete frozen estimator and numerical operators\nThe pretrained model’s documented DDPM schedule uses linear betas from 0.0001 to 0.02 over 1,000 timesteps. For a clean image \(x_0\) and cumulative \(\bar\alpha_t\), the model is evaluated at \(x_t=\sqrt{\bar\alpha_t}x_0\) (zero added random noise). With \(\sigma_t=\sqrt{(1-\bar\alpha_t)/\bar\alpha_t}\), our normalized Hessian is \(Q_t=\sqrt{1-\bar\alpha_t}\operatorname{sym}(J_{x_t}\epsilon_\theta(x_t,t,y{=}1))\). The Jacobian-vector product uses only **two model forward evaluations**; the adjoint uses **one first-order backward pass**, avoiding the earlier attention double-derivative failure.

In [ ]:
%%writefile /content/MNIST_Digit1_Final_Geometry.py
#!/usr/bin/env python3
"""Reproducible single-image MNIST digit-1 feasibility experiment (NOT official LHSD).

Pretrained class-conditional DDPM: harveymannering/mnist-ddpm.
Numerical operator: Q_t = sqrt(1-alpha_bar_t) * sym(J_{x_t} eps_theta),
where x_t=sqrt(alpha_bar_t)*x_clean and conditioning class is 1.
This equals -sigma_VE**2 sym(J_{x_clean} s_theta) with
sigma_VE=sqrt((1-alpha_bar_t)/alpha_bar_t).

Jacobian-vector products use 2 forward passes (central difference); adjoint J^T v
uses a FIRST derivative only, avoiding fused-attention double-backward errors.

Selector constants were frozen in the earlier constructed-score benchmark.
This is NOT the official published LHSD implementation, and MNIST LID has no
known ground-truth dimension. Independent translations/nearest-neighbor tests
provide heuristic geometry checks, not certified tangent ground truth.
"""
import argparse
import hashlib
import json
import math
import os
import time
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F

MODEL_ID = 'harveymannering/mnist-ddpm'
# Frozen constants. No MNIST-label or validation-set-based tuning.
SELECT = dict(transition=0.12, defect=0.24, cross=0.28, asym=0.06, range=0.12,
              spectral_cutoff=0.10, hill_power=4, min_run=2)
SIGMAS_T = [5, 15, 35, 65, 100]


def config_for(mode):
    if mode == 'pilot':
        return dict(timesteps=[5, 35, 100], lanczos_steps=10, trace_probes=2,
                    geom_probes=2, filter_steps=10, fd_eps=0.04, ref_neighbors=60)
    if mode == 'expanded':
        return dict(timesteps=SIGMAS_T, lanczos_steps=20, trace_probes=4,
                    geom_probes=4, filter_steps=20, fd_eps=0.04, ref_neighbors=80)
    raise ValueError(mode)


def alpha_cumprod(t):
    # Author's model card: linear DDPM betas 1e-4..0.02 over 1000 steps
    betas = torch.linspace(0.0001, 0.02, 1000, dtype=torch.float32)
    return float(torch.cumprod(1.0-betas, dim=0)[int(t)])


def model_load(device):
    try:
        from diffusers import UNet2DModel
    except Exception as ex:
        raise RuntimeError('Install diffusers and huggingface_hub first.') from ex
    net = UNet2DModel.from_pretrained(MODEL_ID, use_safetensors=True).to(device).eval()
    for p in net.parameters(): p.requires_grad_(False)
    assert net.config.num_class_embeds == 11, f'Expected conditional checkpoint (11 classes), got {net.config.num_class_embeds}'
    assert net.config.in_channels == 1 and net.config.sample_size == 28, 'Expected 1x28x28 images'
    return net


def image_load(idx):
    from torchvision.datasets import MNIST
    ds = MNIST('/content/mnist_geometry', train=False, download=True)
    assert int(ds.targets[idx]) == 1, f'MNIST test index {idx} has label {int(ds.targets[idx])}, not 1'
    x = ds.data[idx].float()/127.5 - 1.0
    train = MNIST('/content/mnist_geometry', train=True, download=True)
    return x.reshape(1,1,28,28), train


class ScoreOperator:
    """Symmetric part of normalized score Hessian Q, matrix-free."""
    def __init__(self, net, x, t, fd_eps=0.04):
        self.net = net
        self.t = int(t)
        self.ab = alpha_cumprod(t)
        self.sqrt_ab = math.sqrt(self.ab)
        self.scale = math.sqrt(1-self.ab)
        self.sigma = math.sqrt((1-self.ab)/self.ab)
        self.xt = self.sqrt_ab * x.detach().clone()
        self.device = x.device
        self.D = x.numel()
        self.fd_eps = float(fd_eps)
        self.image_forward_evals = 0
        self.backward_evals = 0
        self.sym_calls = 0
        self.raw_calls = 0
        self.adj_calls = 0

    def model_call(self, batch):
        ids=torch.full((batch.shape[0],),self.t,device=self.device,dtype=torch.long)
        labels=torch.ones(batch.shape[0],device=self.device,dtype=torch.long)
        return self.net(batch,ids,class_labels=labels).sample

    def q_raw(self, vector, h=None):
        """Q_raw v by forward differences of epsilon network (only forwards)."""
        self.raw_calls += 1
        v = vector.detach().to(self.xt).reshape_as(self.xt)
        n = float(torch.linalg.vector_norm(v))
        if n < 1e-15: return torch.zeros_like(vector)
        step = (h if h is not None else self.fd_eps)/n
        with torch.no_grad():
            xs=torch.cat([self.xt+step*v,self.xt-step*v],dim=0)
            out=self.model_call(xs)
            self.image_forward_evals += 2
            jv=(out[0:1]-out[1:2])/(2*step)
        return (self.scale*jv).reshape(-1)

    def q_adj(self, vector):
        """Q_raw^T v by first-order reverse-mode gradient, no double backward."""
        self.adj_calls += 1
        v=vector.detach().to(self.xt).reshape_as(self.xt)
        with torch.enable_grad():
            inp=self.xt.detach().clone().requires_grad_(True)
            out=self.model_call(inp)
            scalar=(out*v).sum()
            grad=torch.autograd.grad(scalar,inp,create_graph=False)[0]
        self.image_forward_evals += 1
        self.backward_evals += 1
        return (self.scale*grad).detach().reshape(-1)

    def q_sym(self, vector):
        self.sym_calls += 1
        return 0.5*(self.q_raw(vector)+self.q_adj(vector))

    def counters(self):
        return dict(sym_calls=self.sym_calls,fd_calls=self.raw_calls,
                    vjp_calls=self.adj_calls,model_forward_image_evals=self.image_forward_evals,
                    model_backward_evals=self.backward_evals)


class DiagOp:
    def __init__(self, diag):
        self.diag=torch.as_tensor(diag,dtype=torch.float32)
        self.D=self.diag.numel()
        self.device=self.diag.device
    def q_sym(self,v): return self.diag * v


def lanczos(op, v, nsteps=12):
    z=v.detach().flatten().float().to(op.device)
    mag=float(torch.linalg.vector_norm(z).item())
    if mag < 1e-14: raise ValueError('Near-zero Lanczos start vector')
    z=z/mag
    basis=[]; alphas=[];betas=[]
    prev=torch.zeros_like(z); prev_beta=0.0
    for it in range(min(nsteps,z.numel())):
        w=op.q_sym(z)
        a=float(torch.dot(z,w).item())
        w=w-a*z-prev_beta*prev
        # Two-pass full reorthogonalization for spectral numerical robustness.
        for _ in range(2):
            for q in basis+[z]:
                w=w-torch.dot(q,w)*q
        b=float(torch.linalg.vector_norm(w))
        basis.append(z)
        alphas.append(a)
        if b < 1e-8 or it==nsteps-1: break
        betas.append(b)
        prev=z;prev_beta=b;z=w/b
    T=np.diag(np.asarray(alphas,dtype=np.float64))
    for i,b in enumerate(betas):
        T[i,i+1]=b; T[i+1,i]=b
    eig,U=np.linalg.eigh(T)
    return eig,U,basis,mag


def hill(eig):
    return 1./(1.+(np.abs(eig)/SELECT['spectral_cutoff'])**SELECT['hill_power'])


def spectral_data(op,steps,probes,seed):
    gen=torch.Generator(device='cpu').manual_seed(seed)
    vals=[]
    for p in range(probes):
        z=(torch.randint(0,2,(op.D,),generator=gen,dtype=torch.int64)*2-1).float().to(op.device)
        lam, U, _, _ = lanczos(op,z,steps)
        weights=U[0,:]**2
        deff=float(op.D*np.dot(weights,hill(lam)))
        transition=float(np.dot(weights,( (np.abs(lam)>=.05)&(np.abs(lam)<=.20)).astype(float)))
        violation=float(np.dot(weights,np.maximum(-lam,0)+np.maximum(lam-1,0)))
        vals.append((deff,transition,violation))
    arr=np.asarray(vals)
    return dict(d_eff=float(arr[:,0].mean()), d_round=int(np.rint(arr[:,0].mean())),
                d_probe_SE=float(arr[:,0].std(ddof=1)/math.sqrt(probes)) if probes>=2 else np.nan,
                transition=float(arr[:,1].mean()),range_violation=float(arr[:,2].mean()),
                lanczos_calls_max=steps*probes)


def probe_vectors(D,seed,count,device):
    gen=torch.Generator(device='cpu').manual_seed(seed)
    out=[]
    for _ in range(count):
        z=(torch.randint(0,2,(D,),generator=gen)*2-1).float().to(device)
        out.append(z/math.sqrt(D))
    return out


def geometry_data(ops,probes=2,seed=123):
    # Common probes across ALL scales: reduces comparison noise.
    zlist=probe_vectors(ops[0].D,seed,probes,ops[0].device)
    geo=[]
    for i,op in enumerate(ops):
        numerator=denom=asym_num=asym_den=cnum=cden=0.0
        adj_ids=[j for j in [i-1,i+1] if 0<=j<len(ops)]
        for z in zlist:
            q=op.q_sym(z);qq=op.q_sym(q)
            numerator += float(torch.sum((qq-q)**2))
            denom += float(torch.sum(q**2))
            raw=op.q_raw(z);adj=op.q_adj(z)
            asym_num += float(torch.sum((raw-adj)**2))
            asym_den += .5*float(torch.sum(raw**2)+torch.sum(adj**2))
            for j in adj_ids:
                qr=ops[j].q_sym(z)
                cnum += float(torch.sum((q-qr)**2))
                cden += float(torch.sum(q**2))
        geo.append(dict(projector_defect=math.sqrt(numerator/max(denom,1e-20)),
                        antisymmetry=math.sqrt(asym_num/max(asym_den,1e-20)),
                        cross_scale=math.sqrt(cnum/max(cden,1e-20))))
    return geo


def scale_selection(rows,method):
    # Exact same threshold and contiguous-region rules as previous bridge.
    rows=sorted(rows,key=lambda r:r['sigma'])
    for r in rows:
        r['spectral_valid']=bool(r['transition']<=SELECT['transition'] and 0.5<r['d_eff']<784-0.5)
        r['joint_valid']=bool(r['spectral_valid'] and r['projector_defect']<=SELECT['defect'] and
            r['cross_scale']<=SELECT['cross'] and r['antisymmetry']<=SELECT['asym'] and
            r['range_violation']<=SELECT['range'])
    key='spectral_valid' if method=='LHSD_inspired_transition' else 'joint_valid'
    groups=[];group=[]
    for i,r in enumerate(rows):
        if r[key]: group.append(i)
        elif group: groups.append(group);group=[]
    if group:groups.append(group)
    groups=[g for g in groups if len(g)>=SELECT['min_run']]
    if not groups:return None
    if method=='LHSD_inspired_transition':
        g=min(groups,key=lambda ids:(-len(ids),np.mean([rows[j]['transition'] for j in ids])))
        return rows[g[(len(g)-1)//2]]
    metric=lambda r:r['transition']+r['projector_defect']+r['cross_scale']
    g=min(groups,key=lambda ids:(np.mean([metric(rows[j]) for j in ids]),-len(ids)))
    return rows[min(g,key=lambda j:metric(rows[j]))]


def filter_apply(op, vector, steps):
    v=vector.detach().flatten().float().to(op.device)
    eig,U,basis,mag=lanczos(op,v,steps)
    coeff=(U * (hill(eig)*U[0,:])[None,:]).sum(axis=1)
    out=sum(float(c)*q for c,q in zip(coeff,basis))
    return mag*out


def shifted_image(x, dx, dy):
    # Subpixel shifts (not used for score selection).
    theta=torch.tensor([[[1.,0.,-2*dx/28],[0.,1.,-2*dy/28]]],dtype=x.dtype,device=x.device)
    grid=F.affine_grid(theta,x.size(),align_corners=False)
    return F.grid_sample(x,grid,mode='bilinear',padding_mode='border',align_corners=False)


def independent_vectors(x,train,neighbors=60):
    # Never run before scale selection; independent retrospective evaluation only.
    h=.25
    vx=((shifted_image(x,h,0)-shifted_image(x,-h,0))/(2*h)).flatten()
    vy=((shifted_image(x,0,h)-shifted_image(x,0,-h))/(2*h)).flatten()
    alltrain=train.data[train.targets==1].float().reshape(-1,784)/127.5-1.0
    xcpu=x.cpu().flatten()
    dists=torch.linalg.vector_norm(alltrain-xcpu.unsqueeze(0),dim=1)
    nearest=torch.topk(dists,k=neighbors,largest=False).indices
    delta=(alltrain[nearest]-xcpu.unsqueeze(0))
    _,_,vt=torch.linalg.svd(delta,full_matrices=False)
    vecs={'translation_x':vx,'translation_y':vy}
    for i in range(4):vecs[f'neighbor_pc_{i+1}']=vt[i].to(x.device)
    gen=torch.Generator(device='cpu').manual_seed(90210)
    for i in range(4):vecs[f'random_{i+1}']=torch.randn(784,generator=gen).to(x.device)
    return vecs


def independent_validation(op, vectors, steps, method):
    rows=[]
    for name,vec in vectors.items():
        base=vec.flatten().float().to(op.device)
        v=base/torch.linalg.vector_norm(base).clamp_min(1e-12)
        try:
            projected=filter_apply(op,v,steps)
            retention=float(torch.sum(projected**2))
            reconstruction_error=float(torch.linalg.vector_norm(projected-v))
            cosine=float(torch.dot(projected,v)/torch.linalg.vector_norm(projected).clamp_min(1e-12))
        except Exception as e:
            raise RuntimeError(f'Failed independent validation {name}') from e
        rows.append(dict(method=method,direction=name,squared_retention=retention,
                         reconstruction_error=reconstruction_error,cosine=cosine,
                         vector_norm_original=float(torch.linalg.vector_norm(base))))
    return rows


def derivative_audit(op,seed=7):
    z=probe_vectors(op.D,seed,1,op.device)[0]
    j1=op.q_raw(z,h=0.04); j2=op.q_raw(z,h=0.02)
    rel=float(torch.linalg.vector_norm(j1-j2)/torch.linalg.vector_norm(j2).clamp_min(1e-10))
    # Scalar reciprocity: <u, Jv> = <J^T u, v>.
    u=probe_vectors(op.D,seed+1,1,op.device)[0]
    raw=op.q_raw(z);adj=op.q_adj(u)
    pair=float(abs(torch.dot(u,raw)-torch.dot(adj,z)) /
               max(abs(float(torch.dot(u,raw))), abs(float(torch.dot(adj,z))),1e-6))
    return dict(fd_step_relative_disagreement=rel,adjoint_identity_relative_error=pair)


def plot_results(out,rows,sel,x,ind):
    import matplotlib.pyplot as plt
    d=pd.DataFrame(rows).sort_values('sigma')
    fig,axes=plt.subplots(2,3,figsize=(14,8))
    for ax,k in zip(axes.flat,['d_eff','transition','projector_defect','cross_scale','antisymmetry','range_violation']):
        ax.plot(d['sigma'],d[k],marker='o')
        for s in sel:
            if s['accepted']:
                ax.axvline(s['sigma'],linestyle='--',label=s['method'])
        ax.set_xscale('log');ax.set_title(k);ax.set_xlabel('Effective VE sigma')
        ax.grid(alpha=.3)
    axes.flat[0].legend(fontsize=7)
    fig.suptitle('One held-out MNIST digit 1 | same checkpoint & noise scales')
    fig.tight_layout();fig.savefig(out/'scale_diagnostics.png',dpi=145);plt.close(fig)
    fig,axs=plt.subplots(1,2,figsize=(10,4))
    axs[0].imshow(x.detach().cpu().reshape(28,28),cmap='gray',vmin=-1,vmax=1)
    axs[0].set_title('Held-out real MNIST digit 1');axs[0].axis('off')
    d2=pd.DataFrame(ind)
    if not d2.empty:
        d2=d2.pivot(index='direction',columns='method',values='squared_retention')
        d2.plot(kind='bar',ax=axs[1],rot=45)
        axs[1].set_ylabel('Squared tangent-filter retention')
        axs[1].set_title('Independent direction checks')
    else:axs[1].text(.1,.4,'Both methods abstained\nNo geometry claim',fontsize=12)
    fig.tight_layout();fig.savefig(out/'independent_geometry.png',dpi=145);plt.close(fig)


def make_report(out, rows, selections, independent, audit, cfg, idx, device):
    df=pd.DataFrame(independent)
    lines=['# MNIST digit-1 score-geometry feasibility report','',
           f'- Model: `{MODEL_ID}` (frozen pretrained, class label 1).',
           f'- Held-out MNIST test index: {idx}; input dimension 784; model inference device: {device}.',
           '- Score Jacobian: symmetric part of epsilon-prediction Jacobian; central finite difference + first-order reverse mode.',
           '- Comparator: LHSD-**inspired** transition selector, **NOT official LHSD code**.',
           '- Selector rules and thresholds frozen from earlier synthetic benchmark; translations and neighbor data never used for selection.',
           '- MNIST true intrinsic dimension is **unknown**. A dimension estimate is not an accuracy measurement.','',
           '## Scale selection (unverified dimension values unless independently validated)','',
           '| Method | Status | DDPM timestep | VE-equivalent sigma | Estimated ID (SLQ) | SLQ probe SE |',
           '|---|---|---:|---:|---:|---:|']
    for s in selections:
        lines.append(f"| {s['method']} | {'ACCEPT' if s['accepted'] else 'ABSTAIN'} | " +
                     (f"{s['timestep']} | {s['sigma']:.5f} | {s['d_eff']:.2f} | {s['d_probe_SE']:.2f} |" if s['accepted'] else '— | — | — | — |'))
    lines+=['','## Independent checks','']
    if not df.empty:
        for m,g in df.groupby('method'):
            tr=g[g.direction.str.startswith('translation')]['squared_retention'].mean()
            rand=g[g.direction.str.startswith('random')]['squared_retention'].mean()
            nn=g[g.direction.str.startswith('neighbor')]['squared_retention'].mean()
            lines.append(f'- {m}: translation retention {tr:.4f}; random retention {rand:.4f}; clean-neighbor PC retention {nn:.4f}.')
        lines.append('- Retention is a **heuristic** based on a soft spectral tangent filter and is influenced by estimated dimension; do not interpret as true tangent error.')
    else:lines.append('- Both selectors abstained: no independently tested tangent estimate.')
    lines += ['', '## Numerical correctness (at the middle noise scale)',
              f"- Forward-difference step disagreement: {audit['fd_step_relative_disagreement']:.5g}.",
              f"- First-order adjoint duality residual: {audit['adjoint_identity_relative_error']:.5g}.",
              '- Numerical derivative check: '+('PASS (heuristic 5% tolerance).' if max(audit.values()) < 0.05 else '**WARNING: failed 5% tolerance; do not rely on estimated geometry.**'),
              '- Small residuals support the numerical operator only; they do not establish that the pretrained network has learned manifold geometry.',
              '', '## Scientific interpretation',
              '- If one method has better independently measured retention **at a comparable estimated dimension**, this is preliminary real-image geometry evidence, not MNIST true-ID accuracy.',
              '- If both abstain or both align poorly, report that this checkpoint does not support a credible geometry recovery demonstration at the tested scales.',
              '- More images, trained known-dimension score models, official LHSD replication, finite-difference and SLQ convergence are future work.','']
    (out/'PROPOSAL_REPORT.md').write_text('\n'.join(lines))


def run(args):
    import matplotlib
    matplotlib.use('Agg')
    torch.manual_seed(19);np.random.seed(19)
    torch.set_num_threads(min(4,os.cpu_count() or 2))
    cfg=config_for(args.mode)
    device=args.device if args.device != 'auto' else ('cuda' if torch.cuda.is_available() else 'cpu')
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    x,train=image_load(args.index)
    x=x.float().to(device)
    net=model_load(device)
    ops=[ScoreOperator(net,x,t,cfg['fd_eps']) for t in cfg['timesteps']]
    ops.sort(key=lambda a:a.sigma)
    with torch.no_grad():
        check=ops[0].model_call(ops[0].xt)
        assert check.shape==ops[0].xt.shape and torch.isfinite(check).all(), 'Pretrained conditional model failed inference sanity check'
    rows=[]
    print('Pretrained MNIST-conditional checkpoint loaded; clean test label 1;',len(ops),'scales',flush=True)
    for op in ops:
        tick=time.perf_counter()
        stats=spectral_data(op,cfg['lanczos_steps'],cfg['trace_probes'],seed=7)
        rows.append(dict(timestep=op.t,sigma=op.sigma,**stats,slq_seconds=time.perf_counter()-tick))
        print(f"  t={op.t:3d} sigma={op.sigma:.4f} ID(SLQ)={stats['d_eff']:.2f} transition={stats['transition']:.3f}",flush=True)
    print('Computing projector/cross-scale diagnostics...',flush=True)
    geo=geometry_data(ops,cfg['geom_probes'])
    for row,g in zip(rows,geo): row.update(g)
    selections=[]
    for name in ['LHSD_inspired_transition','Projector_cross_scale']:
        choice=scale_selection(rows,name)
        if choice is None:
            selections.append(dict(method=name,accepted=False,timestep=np.nan,sigma=np.nan,d_eff=np.nan,d_probe_SE=np.nan,reason='No consecutive accepted noise-scale interval'))
        else:
            selections.append(dict(method=name,accepted=True,timestep=choice['timestep'],sigma=choice['sigma'],d_eff=choice['d_eff'],d_probe_SE=choice['d_probe_SE'],reason='Frozen selector passed'))
    print('Frozen selectors:',[(s['method'], 'ABSTAIN' if not s['accepted'] else (s['timestep'],round(s['d_eff'],2))) for s in selections],flush=True)
    mid=ops[len(ops)//2]
    print('Running numerical finite-difference/adjoint audit...',flush=True)
    audit=derivative_audit(mid)
    # Held-out translations/clean-neighbor data created AFTER selection. Cannot affect the decision.
    ind=[]
    if any(s['accepted'] for s in selections):
        vectors=independent_vectors(x,train,neighbors=cfg['ref_neighbors'])
        for s in selections:
            if s['accepted']:
                op=next(o for o in ops if o.t == s['timestep'])
                print(f"Independent validation: {s['method']} at t={op.t}...",flush=True)
                ind.extend([dict(timestep=op.t,sigma=op.sigma,**r) for r in
                            independent_validation(op,vectors,cfg['filter_steps'],s['method'])])
    for row,op in zip(rows,ops):row.update(op.counters())
    df=pd.DataFrame(rows);sd=pd.DataFrame(selections);id_df=pd.DataFrame(ind)
    df.to_csv(out/'all_scale_diagnostics.csv',index=False)
    sd.to_csv(out/'selected_scales.csv',index=False)
    id_df.to_csv(out/'independent_tangent_checks.csv',index=False)
    (out/'numerical_audit.json').write_text(json.dumps(audit,indent=2))
    meta=dict(checkpoint=MODEL_ID,checkpoint_architecture='UNet2DModel num_class_embeds=11',
              test_index=args.index,test_label=1,input_scaling='uint8/127.5-1',
              input_sha256=hashlib.sha256(x.detach().cpu().numpy().tobytes()).hexdigest(),
              class_label_used_by_model=1,noise_schedule='DDPM linear beta 0.0001 to 0.02, T=1000',
              Q='sqrt(1-alpha_bar)*sym(J_xt epsilon_theta(xt,t,label=1))',
              model_evaluated_at='xt=sqrt(alpha_bar)*clean_x (no random test corruption)',
              noise_scale='VE-equivalent sigma=sqrt((1-alpha_bar)/alpha_bar)',
              fd_step=cfg['fd_eps'],selection_thresholds=SELECT,config=cfg,device=device,
              official_LHSD=False,ground_truth_MNIST_dimension=None,
              independent_data_used_only_after_selection=True)
    (out/'protocol.json').write_text(json.dumps(meta,indent=2))
    torch.save(dict(x_clean=x.detach().cpu(), test_index=args.index, model_id=MODEL_ID,
                    selected_timesteps={s['method']:(int(s['timestep']) if s['accepted'] else None) for s in selections},
                    note='Reusable pixel-space input; reload the same pretrained score model and ScoreOperator before curvature/projection.'),
               out/'geometry_state.pt')
    plot_results(out,rows,selections,x,ind)
    make_report(out,rows,selections,ind,audit,cfg,args.index,device)
    print('\nSELECTED SCALES:\n'+sd.to_string(index=False))
    print('\nNumerical audit:',audit)
    print('\nFiles saved:',out.resolve())
    print('Read PROPOSAL_REPORT.md. No real-MNIST dimension ground truth. Not official LHSD.')


class LinearEpsMock(torch.nn.Module):
    """A model used only in the unit test, never as MNIST results."""
    def __init__(self, diag):
        super().__init__()
        self.register_buffer('diag',torch.tensor(diag,dtype=torch.float32).reshape(1,1,28,28))
    def forward(self,x,t,class_labels):
        return SimpleNamespace(sample=x*self.diag)


def self_test():
    print('Running analytic operator, SLQ and soft-filter unit tests (no pretrained checkpoint).')
    t=35; scale=math.sqrt(1-alpha_cumprod(t))
    vals=np.concatenate([np.zeros(120),np.ones(784-120)]).astype(np.float32)
    net=LinearEpsMock(vals/scale)
    x=torch.zeros(1,1,28,28)
    op=ScoreOperator(net,x,t)
    g=torch.Generator().manual_seed(42)
    v=torch.randn(784,generator=g)
    assert torch.allclose(op.q_raw(v),torch.as_tensor(vals)*v,atol=1e-3,rtol=1e-3)
    assert torch.allclose(op.q_adj(v),torch.as_tensor(vals)*v,atol=1e-5,rtol=1e-5)
    d=spectral_data(op,steps=8,probes=6,seed=42)
    # Check tridiagonal reconstruction on known vector and exact diagonal matrix
    dv=DiagOp(vals)
    u=torch.randn(784,generator=g)
    result=filter_apply(dv,u,steps=8)
    expected=torch.as_tensor(hill(vals))*u
    assert torch.allclose(result,expected,atol=3e-4,rtol=3e-4),float(torch.linalg.vector_norm(result-expected))
    print('PASS Q finite difference, adjoint first derivative, Lanczos filter. Estimated analytic soft ID:',round(d['d_eff'],2))


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--mode',choices=['pilot','expanded'],default='pilot')
    p.add_argument('--out',default='/content/mnist_digit1_final_results')
    p.add_argument('--index',type=int,default=2,help='Held-out MNIST digit-1 test index; default 2')
    p.add_argument('--device',choices=['auto','cpu','cuda'],default='auto')
    p.add_argument('--self-test',action='store_true')
    a=p.parse_args()
    if a.self_test:self_test()
    else:run(a)


if __name__=='__main__': main()


## 3 · Numerical unit tests (no model download)
This verifies the operator and Lanczos filter using an analytically known projector with 120 tangent directions in 784 dimensions. The reported 120 is **unit-test geometry**, not a measured MNIST dimension.

In [ ]:
!python /content/MNIST_Digit1_Final_Geometry.py --self-test

## 4 · Pilot: one unseen digit `1` · 3 noise scales
**First real-checkpoint experiment.** The script downloads the pretrained conditional UNet and MNIST if not already cached, freezes all network parameters, and applies both scale selectors to test index 2. No retraining. If both selectors abstain, the notebook reports this explicitly.

In [ ]:
!python /content/MNIST_Digit1_Final_Geometry.py --mode pilot --index 2 --device auto --out /content/mnist_digit1_final_pilot

## 5 · Inspect real-image results and numerical reliability
The selected intrinsic dimensions are **estimates**, not verified truths. Pay attention to `accepted`, `d_probe_SE`, the forward-difference audit, the independent tangent-check figure, and model-evaluation counts.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown

folder=Path('/content/mnist_digit1_final_pilot')
display(Markdown((folder/'PROPOSAL_REPORT.md').read_text()))
display(pd.read_csv(folder/'selected_scales.csv'))
display(pd.read_csv(folder/'all_scale_diagnostics.csv')[
    ['timestep','sigma','d_eff','d_probe_SE','transition','projector_defect',
     'cross_scale','antisymmetry','range_violation','spectral_valid','joint_valid',
     'model_forward_image_evals','model_backward_evals']])
display(Image(filename=str(folder/'scale_diagnostics.png')))
display(Image(filename=str(folder/'independent_geometry.png')))

## 6 · Optional expanded run (recommended for proposal figure)
Turn this on **only after the pilot succeeds**, preferably on GPU. It uses five noise scales, four SLQ probes, 20 Lanczos steps, and a stronger independent-vector filter. This is more computationally expensive; the inference checkpoint is still frozen and the noise-scale selection thresholds remain unchanged.

In [ ]:
RUN_EXPANDED = False  # change to True after checking the pilot

if RUN_EXPANDED:
    import subprocess
    subprocess.run([
        'python','/content/MNIST_Digit1_Final_Geometry.py',
        '--mode','expanded','--index','2','--device','auto',
        '--out','/content/mnist_digit1_final_expanded'
    ],check=True)
    expanded=Path('/content/mnist_digit1_final_expanded')
    display(Markdown((expanded/'PROPOSAL_REPORT.md').read_text()))
    display(pd.read_csv(expanded/'selected_scales.csv'))
    display(Image(filename=str(expanded/'scale_diagnostics.png')))
    display(Image(filename=str(expanded/'independent_geometry.png')))
else:
    print('Expanded evaluation skipped. Set RUN_EXPANDED=True when ready.')

## 7 · Export everything for your proposal and later curvature/projection research

The folder contains the clean image and selected scale as `geometry_state.pt`, plus the exact source `MNIST_Digit1_Final_Geometry.py`. In follow-up work, the function `filter_apply(ScoreOperator(...),v,steps)` is our **candidate tangent-direction filter**. It is **not** a certified nonlinear manifold projection; curvature and proper projection require additional validation.

Files: `all_scale_diagnostics.csv`, `selected_scales.csv`, `independent_tangent_checks.csv`, `numerical_audit.json`, `protocol.json`, `scale_diagnostics.png`, `independent_geometry.png`, `PROPOSAL_REPORT.md`, and `geometry_state.pt`.

In [ ]:
import os, shutil
from pathlib import Path
root=Path('/content')
out=root/'mnist_digit1_final_expanded' if (root/'mnist_digit1_final_expanded'/'selected_scales.csv').exists() else root/'mnist_digit1_final_pilot'
assert out.exists(), 'Run the pilot first.'
shutil.copy2('/content/MNIST_Digit1_Final_Geometry.py', out/'MNIST_Digit1_Final_Geometry.py')
archive=shutil.make_archive(str(root/'MNIST_Digit1_Final_Results'),'zip',root_dir=str(out))
print('Created:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Outside Colab: retrieve the ZIP at the printed path.')

## Interpretation for the PhD proposal

- **If our selector accepts and outperforms the transition-only baseline on independent direction tests**, report a one-image *feasibility observation* and state the selected sigma, estimated dimension, numerical uncertainty, independent direction comparison, and extra evaluation cost. Do **not** call the estimate the true dimension.
- **If both selectors accept with similar results**, report that the method executed successfully but no real-image advantage has yet been shown.
- **If our method abstains**, report that the pretrained score Hessian did not satisfy the frozen geometric criteria at the tested scales. This is a useful reliability failure and does **not** support claiming successful MNIST manifold recovery.

Your earlier *known-dimension synthetic* experiments remain the evidence for dimension-accuracy improvement under controlled score errors. This notebook contributes a **real-image implementation and independent geometry feasibility check**, not a reproduction of official LHSD or verified MNIST intrinsic-dimension accuracy. For the future curvature/projection study, only use the recovered geometry as a hypothesis if the numerical and independent validation checks are satisfactory.

**References:** [Pretrained conditional DDPM](https://huggingface.co/harveymannering/mnist-ddpm); [LHSD, ICML 2026](https://proceedings.mlr.press/v306/osada26a.html); [Author's LHSD repository](https://github.com/geosada/LHSD).